# Open Play Event Detection — 테스트

단일 경기에 대해 OpenPlayEventDetector를 실행하고 결과를 분석한다.

In [1]:
%load_ext autoreload
%autoreload 2

import importlib
import pandas as pd

import tools.config as cfg
import autoevent.pipeline as pl
import autoevent.open as op

importlib.reload(cfg)
importlib.reload(pl)
importlib.reload(op)

from autoevent.pipeline import run_pipeline
from autoevent.open import OpenPlayEventDetector
from tools.evaluate import (
    eval_match, micro_summary,
    prepare_gt, prepare_pred,
    evaluate_paper,
    OPEN_LABELS, PAPER_WINDOW,
)
from tools.sportec_data import SportecData

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

print(f"R_PZ={cfg.R_PZ}  THROW_IN_BALL_Z_MIN={cfg.THROW_IN_BALL_Z_MIN}  SP_LOSS_MAP_WINDOW={cfg.SP_LOSS_MAP_WINDOW}")

R_PZ=2.5  THROW_IN_BALL_Z_MIN=0.5  SP_LOSS_MAP_WINDOW=5


## 1. 경기 선택 및 데이터 로드

In [2]:
MATCH_ID = "J03WMX"  # 변경 가능: J03WMX / J03WN1 / J03WOH / J03WOY / J03WPY / J03WQQ / J03WR9

match = SportecData(MATCH_ID)
result = run_pipeline(match.tracking)
test = OpenPlayEventDetector(result).run()

print(f"{MATCH_ID} 로드 완료 — tracking {len(match.tracking):,}행, pred events {test['event_name'].notna().sum()}개")

Loading the tracking data...
Transforming the tracking data coordinates...


KeyboardInterrupt: 

## 2. 예측 이벤트 분포

In [ ]:
print("=== Pred event_name 분포 ===")
print(test['event_name'].value_counts().to_string())

## 3. 평가 (Open play, ±10s + player)

In [ ]:
gt_eval, gt_sp = prepare_gt(match)
pred_full, pred_sp = prepare_pred(result, test)

open_rows = [evaluate_paper(gt_eval, pred_full, lbl, window=PAPER_WINDOW, use_player=True)
             for lbl in OPEN_LABELS]
open_df = pd.DataFrame(open_rows).set_index("label")

print(f"=== {MATCH_ID} — Open play (±{PAPER_WINDOW}s + player) ===")
print(open_df.to_string())
m = micro_summary(open_df)
print(f"\nMicro  TP={m['TP']} FP={m['FP']} FN={m['FN']}  P={m['Precision']:.3f} R={m['Recall']:.3f} F1={m['F1']:.3f}")

## 4. FN 분석 (GT에 있는데 탐지 못한 이벤트)

In [ ]:
LABEL = "pass"  # 분석할 label: pass / cross / shot / interception

gt_sub   = gt_eval[gt_eval["label"] == LABEL][["period_id","timestamp","object_id"]].reset_index(drop=True)
pred_sub = pred_full[pred_full["label"] == LABEL][["period_id","timestamp","event_player"]].reset_index(drop=True)

matched_gt, matched_pred = set(), set()
for gi, gr in gt_sub.iterrows():
    cand = pred_sub[
        (pred_sub["period_id"] == gr["period_id"]) &
        (abs(pred_sub["timestamp"] - gr["timestamp"]) <= PAPER_WINDOW) &
        (pred_sub["event_player"] == gr["object_id"])
    ]
    cand = cand[~cand.index.isin(matched_pred)]
    if cand.empty:
        continue
    best = (cand["timestamp"] - gr["timestamp"]).abs().idxmin()
    matched_gt.add(gi)
    matched_pred.add(best)

fn_df = gt_sub[~gt_sub.index.isin(matched_gt)].copy()
print(f"=== FN ({LABEL}) — {len(fn_df)}개 ===")
print(fn_df.to_string())

## 5. FP 분석 (예측했는데 GT에 없는 이벤트)

In [ ]:
fp_df = pred_sub[~pred_sub.index.isin(matched_pred)].copy()
print(f"=== FP ({LABEL}) — {len(fp_df)}개 ===")
print(fp_df.to_string())